# MAPPO Full-Layout 60-Ant Half-Food Shared Writes Write-Cost 8-Bit 50x50

Stabilize the preserved eval-selected 60-ant 8-bit shared-write write-cost checkpoint. The policy remains a shared actor with 8 repeating one-hot identity types, assigned by ant index modulo 8.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status

In [ ]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")

## Quick Smoke Run

Run one tiny job before starting the long continuation.

In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics

## Curriculum Settings

Edit `experiments/exploration_to_forage_full_layout_60ants_half_food_50x50_shared_writes_write_cost_8bits_stabilize_from_60best.json` for durable checkpoint, write-cost, layout, reward-scale, or budget changes. This recipe keeps `num_ants=60`, uses `agent_identity_types=8`, lowers the PPO learning rate, disables entropy pressure, resets Adam on load, and evaluates frequently so the preserved policy is not destroyed by continuation training.


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_full_layout_60ants_half_food_50x50_shared_writes_write_cost_8bits_stabilize_from_60best.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / experiment.name
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
PROBE_DIR = RUN_DIR / "probe"
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment.metadata)
WANDB_VIDEO_KEY_PREFIX = str(experiment.metadata["wandb_video_key_prefix"])
WANDB_VIDEO_MAX_FRAMES = int(experiment.metadata["wandb_video_max_frames"])
WANDB_VIDEO_STAGE_NAMES = tuple(experiment.metadata["wandb_preview_stage_names"])
WANDB_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("wandb_preview_rollout_count", 1))
STAGE_UPDATE_MULTIPLIER = float(experiment.metadata.get("stage_update_multiplier", 1.0))
CHECKPOINT_VIDEO_INTERVAL_UPDATES = int(experiment.metadata["checkpoint_video_interval_updates"])
CHECKPOINT_VIDEO_MAX_FRAMES = int(experiment.metadata["checkpoint_video_max_frames"])
CHECKPOINT_VIDEO_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(
    experiment.metadata,
    key="checkpoint_video_policy_temperature",
)
CHECKPOINT_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("checkpoint_video_rollout_count", 1))
CHECKPOINT_VIDEO_RENDER_STYLE = experiment.metadata.get("checkpoint_video_render_style")
CHECKPOINT_VIDEO_SHOW_VISION = bool(experiment.metadata.get("checkpoint_video_show_vision", True))
CHECKPOINT_VIDEO_WANDB_KEY_PREFIX = str(experiment.metadata["checkpoint_video_wandb_key_prefix"])
ROLLOUT_RENDER_STYLE = experiment.metadata.get("rollout_render_style")
ROLLOUT_SHOW_VISION = bool(experiment.metadata.get("rollout_show_vision", True))
SOURCE_CHECKPOINT = None
if experiment_args.get("load_model"):
    SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
    if not SOURCE_CHECKPOINT.exists():
        raise FileNotFoundError(f"Run or restore the source checkpoint first: {SOURCE_CHECKPOINT}")
    experiment_args["load_model"] = str(SOURCE_CHECKPOINT)
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
experiment_args["save_best_model"] = str(BEST_CHECKPOINT_PATH)

SOURCE_COUNTS = tuple(int(count) for count in experiment.metadata["food_source_counts"])
CLUSTER_RADII = tuple(int(radius) for radius in experiment.metadata["food_cluster_radii"])
CURRICULUM_STAGES = workflows.build_food_cluster_curriculum_stages(
    experiment_args,
    source_counts=SOURCE_COUNTS,
    cluster_radii=CLUSTER_RADII,
    visit_reward_schedule=experiment.metadata.get("visit_reward_schedule"),
    view_reward_schedule=experiment.metadata.get("view_reward_schedule"),
    stage_update_multiplier=STAGE_UPDATE_MULTIPLIER,
)
GLOBAL_UPDATE_CAP = max(int(stage["global_update_cap"]) for stage in CURRICULUM_STAGES)
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)
WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = experiment.name
WANDB_RUN_NAME = WANDB_GROUP
WANDB_MODE = "online"
CRITIC_TAG = f"{experiment_args.get('critic_architecture', 'mlp').replace('_', '-')}-critic"
WRITE_BIT_PENALTY = float(experiment_args.get("write_bit_penalty", 0.0))
WRITE_BIT_PENALTY_DECAY = float(experiment_args.get("write_bit_penalty_decay", 0.5))
WRITE_BIT_FULL_VALUE_COST = sum(
    WRITE_BIT_PENALTY * (WRITE_BIT_PENALTY_DECAY ** bit_index)
    for bit_index in range(int(experiment_args["write_bits"]))
)
WRITE_SLOTS_PER_EPISODE = int(experiment_args["num_ants"]) * int(experiment_args["max_steps"])
COMMON_ARGS = workflows.config_common_args(
    experiment_args,
    exclude=workflows.EXPLORATION_TO_FORAGE_ARG_EXCLUDES,
)
{
    "experiment": experiment.name,
    "source_checkpoint": SOURCE_CHECKPOINT,
    "best_checkpoint": BEST_CHECKPOINT_PATH,
    "run_dir": RUN_DIR,
    "layout_margin": experiment_args.get("layout_margin"),
    "hub_center_window_size": experiment_args.get("hub_center_window_size"),
    "macro_food_sources": experiment_args.get("food_cluster_count"),
    "food_count": experiment_args.get("food_count"),
    "num_ants": experiment_args.get("num_ants"),
    "agent_identity_types": experiment_args.get("agent_identity_types"),
    "actor_vision_radius": experiment_args.get("actor_vision_radius"),
    "target_identity_features": experiment.metadata.get("target_identity_features"),
    "actor_identity_policy": experiment.metadata.get("actor_identity_policy"),
    "max_steps": experiment_args.get("max_steps"),
    "critic_architecture": experiment_args.get("critic_architecture"),
    "learning_rate": experiment_args.get("learning_rate"),
    "anneal_lr": experiment_args.get("anneal_lr"),
    "ent_coef": experiment_args.get("ent_coef"),
    "clip_coef": experiment_args.get("clip_coef"),
    "max_grad_norm": experiment_args.get("max_grad_norm"),
    "reset_optimizer_on_load": experiment_args.get("reset_optimizer_on_load"),
    "best_model_selection": experiment_args.get("best_model_selection"),
    "best_eval_interval": experiment_args.get("best_eval_interval"),
    "optimizer_policy": experiment.metadata.get("optimizer_policy"),
    "stabilization_controls": experiment.metadata.get("stabilization_controls"),
    "write_bits": experiment_args.get("write_bits"),
    "per_ant_write_channels": experiment_args.get("per_ant_write_channels"),
    "write_bit_penalty": WRITE_BIT_PENALTY,
    "write_bit_penalty_decay": WRITE_BIT_PENALTY_DECAY,
    "write_head_transfer": experiment_args.get("write_head_transfer"),
    "one_bit_all_slots_episode_cost": WRITE_SLOTS_PER_EPISODE * WRITE_BIT_PENALTY,
    "all_bits_all_slots_episode_cost": WRITE_SLOTS_PER_EPISODE * WRITE_BIT_FULL_VALUE_COST,
    "stage_update_multiplier": STAGE_UPDATE_MULTIPLIER,
    "stage_training_profiles": [
        (
            stage["name"],
            stage["food_sources"],
            stage["food_cluster_radius"],
            stage["food_count"],
            stage["global_update_cap"],
            stage["num_steps"],
            stage["gamma"],
        )
        for stage in CURRICULUM_STAGES
    ],
    "total_updates_per_stage": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
    "checkpoint_video_interval_updates": CHECKPOINT_VIDEO_INTERVAL_UPDATES,
    "checkpoint_video_rollout_count": CHECKPOINT_VIDEO_ROLLOUT_COUNT,
    "checkpoint_video_render_style": CHECKPOINT_VIDEO_RENDER_STYLE,
    "checkpoint_video_show_vision": CHECKPOINT_VIDEO_SHOW_VISION,
    "checkpoint_video_wandb_key_prefix": CHECKPOINT_VIDEO_WANDB_KEY_PREFIX,
    "rollout_render_style": ROLLOUT_RENDER_STYLE,
    "rollout_show_vision": ROLLOUT_SHOW_VISION,
}


## Transfer and Identity Check

Validate that the preserved 60-ant 8-bit checkpoint can be loaded into the stabilization target. The actor observation tail should keep 8 repeating one-hot identity slots, not one unique slot per ant.


In [ ]:
from ant_byte_env.training.jax_mappo.checkpointing import read_checkpoint
from ant_byte_env.training.jax_mappo.transfer import (
    actor_obs_dim_for_bits,
    agent_identity_feature_count,
    load_checkpoint_for_training,
)

FIRST_STAGE = CURRICULUM_STAGES[0]
TRANSFER_CHECK_ARGS = [
    *COMMON_ARGS,
    "--total-timesteps",
    str(UPDATE_TIMESTEPS),
    "--width",
    str(FIRST_STAGE["width"]),
    "--height",
    str(FIRST_STAGE["height"]),
    "--food-count",
    str(FIRST_STAGE["food_count"]),
    "--food-sources",
    str(FIRST_STAGE["food_sources"]),
    "--cookie-distance",
    str(FIRST_STAGE["cookie_distance"]),
    "--max-steps",
    str(FIRST_STAGE["max_steps"]),
]
for key, option in (
    ("num_steps", "--num-steps"),
    ("gamma", "--gamma"),
    ("food_cluster_count", "--food-cluster-count"),
    ("food_cluster_radius", "--food-cluster-radius"),
    ("random_ant_spawn_radius", "--random-ant-spawn-radius"),
):
    if key in FIRST_STAGE:
        TRANSFER_CHECK_ARGS.extend([option, str(FIRST_STAGE[key])])

parsed_args, target_central_obs_dim, target_actor_obs_dim = workflows.training_dimensions(
    TRANSFER_CHECK_ARGS
)
source_checkpoint_data = read_checkpoint(SOURCE_CHECKPOINT)
transferred_checkpoint = load_checkpoint_for_training(
    SOURCE_CHECKPOINT,
    central_obs_dim=target_central_obs_dim,
    actor_obs_dim=target_actor_obs_dim,
    target_write_bits=parsed_args.write_bits,
    actor_vision_radius=parsed_args.actor_vision_radius,
    target_num_ants=parsed_args.num_ants,
    target_agent_identity_types=parsed_args.agent_identity_types,
    write_head_transfer=parsed_args.write_head_transfer,
    target_critic_architecture=parsed_args.critic_architecture,
)
source_num_ants = int(source_checkpoint_data.get("args", {}).get("num_ants", 1))
expected_actor_obs_dim = actor_obs_dim_for_bits(
    write_bits=parsed_args.write_bits,
    actor_vision_radius=parsed_args.actor_vision_radius,
    num_ants=parsed_args.num_ants,
    agent_identity_types=parsed_args.agent_identity_types,
)
if expected_actor_obs_dim != target_actor_obs_dim:
    raise AssertionError("Target actor observation dimension does not include the expected one-hot identity width.")
{
    "source_num_ants": source_num_ants,
    "target_num_ants": int(parsed_args.num_ants),
    "source_identity_features": agent_identity_feature_count(source_num_ants),
    "target_identity_features": agent_identity_feature_count(parsed_args.num_ants, agent_identity_types=parsed_args.agent_identity_types),
    "source_actor_obs_dim": int(source_checkpoint_data["actor_obs_dim"]),
    "target_actor_obs_dim": int(target_actor_obs_dim),
    "transferred_actor_obs_dim": int(transferred_checkpoint["actor_obs_dim"]),
    "source_central_obs_dim": int(source_checkpoint_data["central_obs_dim"]),
    "target_central_obs_dim": int(target_central_obs_dim),
    "transferred_central_obs_dim": int(transferred_checkpoint["central_obs_dim"]),
}


## Train 60-Ant Stabilization


In [ ]:
training_result = workflows.run_forage_curriculum(
    stages=CURRICULUM_STAGES,
    checkpoint_dir=CHECKPOINT_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
    initial_checkpoint=SOURCE_CHECKPOINT,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_run_name=WANDB_RUN_NAME,
    wandb_mode=WANDB_MODE,
    wandb_tags=[
        "exploration-to-forage",
        "full-layout-randomization",
        "warm-start",
        "repeating-one-hot-identity",
        "stabilization",
        "optimizer-reset",
        "low-lr",
        "zero-entropy",
        "60-ants",
        "half-food",
        "two-sources",
        "moving-writes",
        "8-write-bits",
        "shared-write-space",
        "write-bit-cost",
        f"write-bit-penalty-{WRITE_BIT_PENALTY:g}",
        CRITIC_TAG,
        "50x50",
    ],
    wandb_notes=experiment.metadata["notes"],
    wandb_artifact_paths=[EXPERIMENT_CONFIG],
    wandb_artifact_prefix="exploration-to-forage-full-layout-60ants-half-food-shared-writes-write-cost-8bits-stabilize",
    checkpoint_name_prefix=experiment_args["exp_name"],
    wandb_video_key_prefix=WANDB_VIDEO_KEY_PREFIX,
    wandb_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
    wandb_video_stage_names=WANDB_VIDEO_STAGE_NAMES,
    wandb_video_policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
    wandb_video_rollout_count=WANDB_VIDEO_ROLLOUT_COUNT,
    checkpoint_video_interval_updates=CHECKPOINT_VIDEO_INTERVAL_UPDATES,
    checkpoint_video_max_frames=CHECKPOINT_VIDEO_MAX_FRAMES,
    checkpoint_video_policy_temperature=CHECKPOINT_VIDEO_POLICY_TEMPERATURE,
    checkpoint_video_rollout_count=CHECKPOINT_VIDEO_ROLLOUT_COUNT,
    checkpoint_video_wandb_key_prefix=CHECKPOINT_VIDEO_WANDB_KEY_PREFIX,
    checkpoint_video_render_style=CHECKPOINT_VIDEO_RENDER_STYLE,
    checkpoint_video_show_vision=CHECKPOINT_VIDEO_SHOW_VISION,
)
FINAL_CHECKPOINT_PATH = training_result["final_checkpoint_path"]
ROLLOUT_CHECKPOINT_PATH = FINAL_CHECKPOINT_PATH
training_result


## Communication Probe

In [ ]:
from ant_byte_env.training.jax_mappo import probe_communication_checkpoint

probe_result = probe_communication_checkpoint(
    Path(ROLLOUT_CHECKPOINT_PATH),
    output_dir=PROBE_DIR,
    num_episodes=4,
    render_rollouts=False,
)
sampled_histogram = probe_result["sampled"]["write_action_histogram"]
deterministic_histogram = probe_result["deterministic"]["write_action_histogram"]
{
    "probe_path": probe_result["probe_path"],
    "sampled_delivery": probe_result["sampled"]["delivery_metrics"],
    "deterministic_delivery": probe_result["deterministic"]["delivery_metrics"],
    "sampled_nonzero_writes": sum(
        count for value, count in sampled_histogram.items() if value != "0"
    ),
    "deterministic_nonzero_writes": sum(
        count for value, count in deterministic_histogram.items() if value != "0"
    ),
    "sampled_bit_activation_rates": probe_result["sampled"]["per_bit_activation_rates"],
    "deterministic_bit_activation_rates": probe_result["deterministic"]["per_bit_activation_rates"],
}


## Optional Local Render and Vault

In [ ]:
# rollout_result = workflows.render_jax_checkpoint_rollout(
#     run_dir=RUN_DIR,
#     checkpoint_path=ROLLOUT_CHECKPOINT_PATH,
#     media_dir=MEDIA_DIR,
#     rollout_filename="jax_mappo_full_layout_60ants_half_food_shared_writes_write_cost_8bits_stabilized_rollout.mp4",
#     title="JAX MAPPO full-layout 60-ant half-food shared-writes write-cost 8-bit stabilized rollout",
#     description="Sampled rollout from the stabilized full-layout 60-ant half-food shared-writes write-cost 8-bit checkpoint.",
#     metadata={
#         "experiment_config": str(EXPERIMENT_CONFIG),
#         "source_checkpoint": str(SOURCE_CHECKPOINT),
#         "best_checkpoint": str(BEST_CHECKPOINT_PATH),
#         "layout_margin": experiment_args.get("layout_margin"),
#         "hub_center_window_size": experiment_args.get("hub_center_window_size"),
#         "macro_food_sources": experiment_args.get("food_cluster_count"),
#         "critic_architecture": experiment_args.get("critic_architecture"),
#         "food_count": experiment_args.get("food_count"),
#         "num_ants": experiment_args.get("num_ants"),
#         "write_bit_penalty": experiment_args.get("write_bit_penalty"),
#         "write_bit_penalty_decay": experiment_args.get("write_bit_penalty_decay"),
#         "food_source_counts": [stage["food_sources"] for stage in CURRICULUM_STAGES],
#         "food_cluster_radii": [stage["food_cluster_radius"] for stage in CURRICULUM_STAGES],
#     },
#     tile_size=ROLLOUT_TILE_SIZE,
#     policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
#     render_style=ROLLOUT_RENDER_STYLE,
#     show_vision=ROLLOUT_SHOW_VISION,
#     wandb_project=WANDB_PROJECT,
#     wandb_entity=WANDB_ENTITY,
#     wandb_group=WANDB_GROUP,
#     wandb_run_name=f"{WANDB_GROUP}_rollout",
#     wandb_mode="disabled",
#     wandb_video_key=None,
#     wandb_step=training_result["stage_metrics"][-1].get("curriculum_global_step"),
# )
# rollout_result
